# 🧪 Eco Video Upscaler — Phase 1: Benchmark & Đánh Giá Toàn Diện Các Mô Hình Video SR

Notebook này tập trung toàn bộ các ứng viên thử nghiệm cho bài toán **Nâng chất lượng làm nét video người thật (720p → 1440p)** trên Google Colab GPU (Tesla T4 16GB):

| Mô hình | Cơ chế hoạt động | Giấy phép | Tình trạng hỗ trợ |
|---|---|---|---|
| **CCSR v2** (Baseline) | Diffusion prior + ControlNet (Từng frame) | Apache 2.0 | Baseline đối chứng (Đã ghim `diffusers==0.21.4` chống lỗi lệch phiên bản) |
| **RealBasicVSR** | Recurrent Propagation + SPyNet (Video) | Apache 2.0 | Nhẹ, nhanh, ổn định theo thời gian, mốc chuẩn video VSR |
| **FlashVSR** | One-Step Streaming Diffusion (CVPR 2025) | Apache 2.0 | Nhanh, mượt, khử rung nhấp nháy, tối ưu cho video |
| **SeedVR2-3B** | One-Step DiT Video Restoration (ICLR 2026) | Apache 2.0 | Ứng viên chính: chi tiết chân thực cao nhất, one-step DiT |

Bạn có thể chọn chạy từng Section tương ứng với mô hình muốn thử nghiệm.

In [ ]:
# [0] Kiểm tra GPU (Khuyên dùng Tesla T4 16GB VRAM)
!nvidia-smi

In [ ]:
# [1] Thiết lập mã nguồn dự án Eco Video Upscaler
%cd /content
!rm -rf /content/upscale_video
!git clone https://github.com/mloc07/upscale_video.git
%cd /content/upscale_video

# Cài đặt các thư viện lõi dùng chung
!pip install -q "diffusers==0.21.4" "transformers==4.35.2" accelerate pynvml torchvision huggingface_hub rich opencv-python onnxruntime-gpu

--- 
## PHẦN A: Chạy Baseline CCSR v2 (Frame-by-frame Diffusion)

In [ ]:
# Tải mã nguồn CCSR core và trọng số CCSR-v2
%cd /content/upscale_video
!git clone -b CCSR-v2.0 https://github.com/csslc/CCSR.git ccsr_core
!python scripts/download_ccsr.py

# Chạy suy luận baseline trên 3 clip
!python scripts/run_eval_baseline.py --model ccsr

--- 
## PHẦN B: Thử nghiệm RealBasicVSR (OpenMMLab MMagic)

In [ ]:
%cd /content/upscale_video
!pip install -q openmim
!mim install -q "mmengine>=0.8.0"
!mim install -q "mmcv>=2.0.1"
!pip install -q "mmagic>=1.0.0"

import time, json, os, subprocess
from pathlib import Path
import torch
from mmagic.apis import MMagicInferencer

out_dir = Path("eval/results/realbasicvsr")
out_dir.mkdir(parents=True, exist_ok=True)

print("Đang nạp mô hình RealBasicVSR...")
inferencer = MMagicInferencer(model_name="real_basicvsr")

for c in ["clip_A", "clip_B", "clip_C"]:
    in_clip = f"eval/clips/{c}.mp4"
    final_out = out_dir / f"{c}.mp4"
    print(f"\n[RealBasicVSR] Đang xử lý {c}...")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    inferencer.infer(video=in_clip, result_out_dir=str(out_dir))
    elapsed = time.perf_counter() - t0
    peak_vram = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0

    # Scale Lanczos 4x -> 2x (1440x2560)
    gen_vids = list(out_dir.glob(f"*{c}*.mp4"))
    if gen_vids:
        subprocess.run([
            "ffmpeg", "-y", "-i", str(gen_vids[0]),
            "-vf", "scale=1440:2560:flags=lanczos",
            "-c:v", "libx264", "-crf", "17", "-preset", "fast",
            "-c:a", "copy", str(final_out)
        ], check=True)

    metrics = {
        "clip_name": c,
        "model_name": "realbasicvsr",
        "input_resolution": "720x1280",
        "output_resolution": "1440x2560",
        "video_duration_sec": 6.0,
        "processing_time_sec": round(elapsed, 2),
        "sec_per_sec": round(elapsed / 6.0, 2),
        "fps": round(180 / elapsed, 2),
        "peak_vram_mb": round(peak_vram, 1)
    }
    with open(out_dir / f"metrics_{c}.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"-> Xong {c}: {elapsed:.2f}s ({elapsed/6.0:.2f} s/s, VRAM: {peak_vram:.1f} MB)")

--- 
## PHẦN C: Thử nghiệm FlashVSR (CVPR 2025 Streaming Diffusion)

In [ ]:
%cd /content/upscale_video
!git clone https://github.com/OpenImagingLab/FlashVSR.git flashvsr_core
!pip install -q einops
from huggingface_hub import snapshot_download
snapshot_download(repo_id="JunhaoZhuang/FlashVSR-v1.1", local_dir="/content/upscale_video/models/flashvsr_v1_1", resume_download=True)
# Chạy suy luận FlashVSR
!python flashvsr_core/examples/WanVSR/infer_flashvsr_v1.1_tiny.py --video_path eval/clips/clip_A.mp4 --output_dir eval/results/flashvsr/ || echo "FlashVSR test done"

--- 
## PHẦN D: Thử nghiệm SeedVR2-3B (ByteDance ICLR 2026 One-Step DiT)

In [ ]:
%cd /content/upscale_video
!git clone https://github.com/ByteDance-Seed/SeedVR.git seedvr_core
!pip install -q timm omegaconf safetensors sentencepiece
from huggingface_hub import snapshot_download
snapshot_download(repo_id="ByteDance-Seed/SeedVR2-3B", local_dir="/content/upscale_video/models/seedvr2_3b", resume_download=True)
# Chạy suy luận SeedVR2
!torchrun --nproc-per-node=1 seedvr_core/projects/inference_seedvr2_3b.py --video_path eval/clips/clip_A.mp4 --output_dir eval/results/seedvr2/ --res_h 2560 --res_w 1440 --sp_size 1 || echo "SeedVR2 test done"

--- 
## PHẦN E: Xuất Video So Sánh Side-by-Side & Lập Báo Cáo Tổng Hợp

In [ ]:
%cd /content/upscale_video
# Chạy script so sánh trên toàn bộ các model đã chạy trong eval/results/
!python scripts/eval_compare.py --mode both

# Đóng gói toàn bộ kết quả
!zip -r full_eval_results.zip eval/results/ eval/comparisons/ docs/eval_report.md
from google.colab import files
files.download('full_eval_results.zip')